## Agent Engine Evaluation

In [ ]:
#Should restart kernel after installation finished
!pip uninstall -y -q vertexai
!pip install --upgrade -q google-cloud-aiplatform[adk,agent_engines]
!pip install --upgrade --force-reinstall -q google-cloud-aiplatform[evaluation]
!pip install --upgrade -q google-adk==1.26.0
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
#Due to Jupyter issue, isolate rendering function from sdk
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import json
import html
from vertexai._genai import types
import vertexai._genai._evals_visualization as sdk
def display_evaluation_dataset(eval_dataset_obj: types.EvaluationDataset) -> None:
    """Displays an evaluation dataset in an IPython environment."""
    from IPython import display

    processed_rows = []
    df = eval_dataset_obj.eval_dataset_df

    for _, row in df.iterrows():
        processed_row = {}
        for col_name, cell_value in row.items():
            if col_name in ["prompt", "request", "response"]:
                processed_row[col_name] = sdk._extract_text_and_raw_json(cell_value)
            elif col_name == "rubric_groups":
                # Special handling for rubric_groups to keep it as a dict
                if isinstance(cell_value, dict):
                    processed_row[col_name] = {
                        k: [
                            (
                                v_item.model_dump(mode="json")
                                if hasattr(v_item, "model_dump")
                                else v_item
                            )
                            for v_item in v
                        ]
                        for k, v in cell_value.items()
                    }
                else:
                    processed_row[col_name] = cell_value
            else:
                if isinstance(cell_value, (dict, list)):
                    processed_row[col_name] = json.dumps(
                        cell_value, ensure_ascii=False, default=sdk._pydantic_serializer
                    )
                else:
                    processed_row[col_name] = cell_value
        processed_rows.append(processed_row)

    dataframe_json_string = json.dumps(processed_rows, ensure_ascii=False, default=str)
    html_content = sdk._get_inference_html(dataframe_json_string)
    #print(html_content)
    escaped_html = html.escape(html_content)
    iframe_code = f'<div><iframe srcdoc="{escaped_html}" width="100%" height="600px" style="border:none;"></iframe></div>'
    display.display(display.HTML(iframe_code))

In [ ]:
import google.auth
credentials, PROJECT_ID = google.auth.default()
PROJECT_ID

In [ ]:
AGENT_ENGINE_ID = "0000000000000000000"

In [ ]:
import time
import vertexai
from vertexai import Client
from google.genai import types as genai_types
from typing import Optional

import sys
sys.path.append('../app/')

LOCATION = "us-central1"
STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-engine"
REASONING_ENGINE_ID = f"projects/{PROJECT_ID}/locations/{LOCATION}/reasoningEngines/{AGENT_ENGINE_ID}"
GCS_DEST = f"{STAGING_BUCKET}/output-path"
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
)

client = Client(
    project=PROJECT_ID,
    location=LOCATION,
    http_options=genai_types.HttpOptions(api_version="v1beta1"),
)

Evaluate deployed agent

In [ ]:
#Prepare sample data
import pandas as pd
from vertexai._genai import types

session_inputs = types.evals.SessionInput(
    user_id="Test user",
    state={},
)
agent_prompts = [
    "치킨 커리 레시피가 궁금해요",
    "1주일 채식주의자를 위한 식단을 알려주세요",
]
agent_dataset = pd.DataFrame({
    "prompt": agent_prompts,
    "session_inputs": [session_inputs] * len(agent_prompts),
})
agent_dataset

In [ ]:
#Get response
agent_dataset_with_inference = client.evals.run_inference(
    agent=REASONING_ENGINE_ID,
    src=agent_dataset,
)

In [ ]:
display_evaluation_dataset(agent_dataset_with_inference)

In [ ]:
from vertexai._genai import types
from agent import root_agent

root_agent.tools = []

agent_info = types.evals.AgentInfo.load_from_agent(
    root_agent,
    REASONING_ENGINE_ID
)

In [ ]:
evaluation_run = client.evals.create_evaluation_run(
    dataset=agent_dataset_with_inference,
    agent_info=agent_info,
    metrics=[
        types.RubricMetric.FINAL_RESPONSE_QUALITY,
        types.RubricMetric.TOOL_USE_QUALITY,
        types.RubricMetric.HALLUCINATION,
        types.RubricMetric.SAFETY,
    ],
    dest=GCS_DEST,
    config=types.CreateEvaluationRunConfig(
        http_options=genai_types.HttpOptions(
            retry_options=genai_types.HttpRetryOptions(
                attempts=5,           # 최대 재시도 횟수
                initial_delay=1.0,    # 첫 대기 시간
                http_status_codes=[429, 500, 502, 503, 504] # 재시도 대상 에러 코드
            )
        )
    )
)

In [ ]:
evaluation_run = client.evals.get_evaluation_run(
    name=evaluation_run.name,
    include_evaluation_items=True
)

evaluation_run.show()

In [ ]:
while evaluation_run.state not in {"SUCCEEDED", "FAILED", "CANCELLED"}:
    evaluation_run.show()
    evaluation_run = client.evals.get_evaluation_run(name=evaluation_run.name)
    time.sleep(5)

evaluation_run = client.evals.get_evaluation_run(
    name=evaluation_run.name, include_evaluation_items=True
)

# Display the Evaluation Run status and results
evaluation_run.show()